# Skill extraction: CV skills to the KB

Maps the English skill strings of each CV to skill labels of the KB set in `KB` (`"ESCO"` or
`"ONET"`) with `KBMapper` (`kb_mapper/`): exact label or alias match, else embedding retrieval
(top 12) and an LLM rerank (`qwen3:8b`) that may answer `NO_MATCH`. The CV's job category is
passed as `context` and is part of the mapper's cache key, so keep it coarse.

`KB` selects the graph cache, the aliases and descriptions in `sources/{KB}/processed/`, and the
output file names.

- In: `mapping/translated_skills.csv` from `1_translate_skills.ipynb`, joined with `job_category`
  from `mapping/preprocessed_cvs.json`. The input must be English: on Spanish strings the mapper
  returns wrong matches (`Ventas` to `Moodle`, `Atención al cliente` to `write Spanish`).
- Target labels: KB skills at level 4 or deeper (`skill_pool_uris(min_level=4)`).
- Out: `mapping/{KB}_skill_map.csv`, one row per CV (`raw_skills`, the English input, and
  `esco_skills`), merged in `3_build_cvs.ipynb`; `mapping/{KB}_skill_map_detailed.csv`, one row
  per raw skill with score, rank, candidates and `decision_source` (`EXACT`, `LLM_RERANK`,
  `NO_MATCH`). The column is called `esco_skills` for every KB; `3_build_cvs.ipynb` renames it.

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import sys; sys.path.append("../../../"); sys.path.append("../../utils")
import pandas as pd

from ast import literal_eval
from pathlib import Path

from kb_graph import KBGraph
from kb_mapper import KBMapper, skill_pool_uris
from mapping_utils import pending, run_chunked

# KB to map onto: sets the graph cache, the mapper's aliases and descriptions, the output files.
KB = "ONET"
GRAPH_CACHE = {"ESCO": "occSkill_graph.pkl", "ONET": "onet_occSkill_graph.pkl"}[KB]

In [ ]:
translated = pd.read_csv("mapping/translated_skills.csv", converters={"eng_skills": literal_eval})
preprocessed_cvs = (
    pd.read_json("mapping/preprocessed_cvs.json")[["id_c", "job_category"]]
    .merge(translated[["id_c", "eng_skills"]], on="id_c")
)
preprocessed_cvs.head(10)

In [ ]:
kb_g = KBGraph.load(Path("../../../cache") / GRAPH_CACHE)

all_skills_uri = skill_pool_uris(kb_g, min_level = 4)
all_skills = sorted(kb_g.get_label(i) for i in all_skills_uri)

assert all(kb_g.get_level(i) >= 4 for i in all_skills_uri)
assert all(kb_g.get_level(kb_g.get_uri(i, reduce="Down")) >= 4 for i in all_skills)

In [ ]:
MAPPER = KBMapper(all_skills, node="skill", processed_dir=Path("../../../sources") / KB / "processed", kb=KB)

In [ ]:
def extract_skills(row):
    """Map every skill of the CV onto the KB. Returns the aggregate row and N audit rows."""
    mapped = [(skill, MAPPER.map_skill(skill, context=str(row["job_category"])))
              for skill in row["eng_skills"]]
    return (
        {"id_c": row["id_c"], "raw_skills": row["eng_skills"],
         "esco_skills": sorted({r["kb_label"] for _, r in mapped if r["kb_label"]})},
        [{"id_c": row["id_c"], "raw_skill": skill, **r} for skill, r in mapped],
    )

In [ ]:
AGG = (f"mapping/{KB}_skill_map.csv", ["id_c", "raw_skills", "esco_skills"])

DET = (f"mapping/{KB}_skill_map_detailed.csv",
       ["id_c", "raw_skill", "kb_label", "semantic_score", "semantic_rank",
        "candidate_list", "decision_source", "confidence", "mapping_reason"])

todo = pending(preprocessed_cvs, AGG, DET)

In [ ]:
run_chunked(todo, extract_skills, AGG, DET)